# 03. Directional Controls & Multi-Random Placebo Distribution (N=20)
This notebook evaluates $+v_{\text{steer}}$, $-v_{\text{steer}}$, and 20 isotropic Gaussian random vectors with 20 published seeds.

### Objectives:
1. Confirm $\|v_{\text{rand}}^{(r)}\|_2 = 1.0$ unit-norm for all random vectors.
2. Evaluate per-question outputs across exact 500 test questions.
3. Compute empirical randomization test $p$-value: $p = \frac{1 + \#\{r: S_r \ge S_{\text{steer}}\}}{R+1}$.
4. Compute hierarchical bootstrap 95% Confidence Intervals.

In [ ]:
# Cell 1: Environment Setup & Thư viện
!pip install -q evaluate bert_score bitsandbytes accelerate transformers pandas scipy
import os, sys, json, time, torch, numpy as np, pandas as pd
from tqdm import tqdm
from scipy.stats import binomtest
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import evaluate
print('✅ PyTorch Version:', torch.__version__)


In [ ]:
# Cell 2: Dataset Search & Data Split
possible_paths = [
    '/kaggle/input/datasets/thanhtranguyn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/trungkiennnn/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese-medical-halueval-15k/vietnamese_medical_halueval_15k_specialized.json',
    '/kaggle/input/vietnamese_medical_halueval_15k_specialized.json',
    'e:/Paper_Steering_VN_15K/data/vietnamese_medical_halueval_15k_specialized.json',
    './data/vietnamese_medical_halueval_15k_specialized.json',
    './vietnamese_medical_halueval_15k_specialized.json'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    search_root = '/kaggle/input' if os.path.exists('/kaggle/input') else '.'
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if file.endswith('.json') and ('halueval' in file.lower() or '15k' in file.lower() or 'medical' in file.lower() or 'vnese' in file.lower()):
                data_path = os.path.join(root, file)
                break
        if data_path: break

print('✅ Resolved Dataset Path:', data_path)
with open(data_path, 'r', encoding='utf-8') as f: full_dataset = json.load(f)
test_data = full_dataset[-500:]
train_pool = full_dataset[:-2205]
print('Total dataset size:', len(full_dataset), '| Test size:', len(test_data))
assert len(test_data) == 500, 'Test size must be 500'


In [ ]:
# Cell 3: Model & Steering Vector Extraction
model_id = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.eval()
bertscore = evaluate.load('bertscore')

pos_acts, neg_acts = [], []
for item in train_pool[:300]:
    q, pos_ans, neg_ans = item['question'], item.get('right_answer', item.get('positive_answer')), item['hallucinated_answer']
    t_pos = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{pos_ans}'
    t_neg = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{neg_ans}'
    with torch.no_grad():
        inp_p = tokenizer(t_pos, return_tensors='pt').to(model.device)
        pos_acts.append(model(inp_p.input_ids, output_hidden_states=True).hidden_states[8][0, -1, :].detach().cpu())
        inp_n = tokenizer(t_neg, return_tensors='pt').to(model.device)
        neg_acts.append(model(inp_n.input_ids, output_hidden_states=True).hidden_states[8][0, -1, :].detach().cpu())
v_diff = torch.stack(pos_acts).mean(dim=0) - torch.stack(neg_acts).mean(dim=0)
v_steer = v_diff / v_diff.norm(p=2)
print('✅ Steering Vector v_steer Extracted!')


In [ ]:
# Cell 4: Generate 20 Isotropic Unit-Norm Random Vectors with 20 Seeds
RANDOM_SEEDS = [42, 101, 202, 303, 404, 505, 606, 707, 808, 909, 111, 222, 333, 444, 555, 666, 777, 888, 999, 1234]
random_vectors = {}
for seed in RANDOM_SEEDS:
    gen = torch.Generator().manual_seed(seed)
    v_rand = torch.randn(v_steer.shape, generator=gen)
    v_rand = v_rand / v_rand.norm(p=2)
    assert abs(float(v_rand.norm(p=2)) - 1.0) < 1e-5, f'Unit norm check failed for seed {seed}'
    random_vectors[f'rand_seed_{seed}'] = v_rand
print(f'✅ Generated {len(random_vectors)} unit-norm random vectors with published seeds!')


In [ ]:
# Cell 5: Safe Hook Creator (step_counter list, no nonlocal error)
def make_directional_hook(v_target, alpha_0=18.0, K=16):
    step_counter = [0]
    def hook(module, inp, out):
        step_counter[0] += 1
        step = step_counter[0]
        alpha_t = alpha_0 * (1.0 - (step - 1) / K) if 1 <= step <= K else 0.0
        if alpha_t != 0.0:
            cur = out[0] if isinstance(out, tuple) else out
            v_curr = v_target.to(device=cur.device, dtype=cur.dtype)
            mod = cur + alpha_t * v_curr
            return (mod,) + out[1:] if isinstance(out, tuple) else mod
        return out
    return hook
print('✅ Directional Hook Function Ready!')


In [ ]:
# Cell 6: Run Directional Control Evaluation Loop (+v_steer, -v_steer, and 20 Random Vectors)
target_layer = model.model.layers[8]
records = []

conditions_to_eval = [('+v_steer', v_steer), ('-v_steer', -v_steer)] + [(k, v) for k, v in random_vectors.items()]

for cond_name, v_target in conditions_to_eval:
    print(f'🚀 Evaluating Directional Condition: {cond_name}...')
    for idx, item in enumerate(tqdm(test_data, desc=f'Evaluating {cond_name}')):
        prompt = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n"
        inp = tokenizer(prompt, return_tensors='pt').to(model.device)
        p_len = inp.input_ids.shape[1]
        
        h_handle = target_layer.register_forward_hook(make_directional_hook(v_target))
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        h_handle.remove()
        
        gen_toks = out[0][p_len:]
        gen_text = tokenizer.decode(gen_toks, skip_special_tokens=True)
        records.append({
            'question_id': f'Q-{idx:03d}',
            'condition': cond_name,
            'generated_text': gen_text,
            'gold_reference': item.get('right_answer', item.get('positive_answer')),
            'hallucinated_reference': item['hallucinated_answer']
        })

df_dir = pd.DataFrame(records)
print(f'✅ Completed generation for {len(df_dir)} total records!')


In [ ]:
# Cell 7: Compute BERTScores, Randomization Test P-Value & Save CSV Outputs
print('Computing BERTScores...')
bs_p = bertscore.compute(predictions=df_dir['generated_text'].tolist(), references=df_dir['gold_reference'].tolist(), model_type='bert-base-multilingual-cased')['f1']
bs_n = bertscore.compute(predictions=df_dir['generated_text'].tolist(), references=df_dir['hallucinated_reference'].tolist(), model_type='bert-base-multilingual-cased')['f1']
df_dir['BS_positive'] = bs_p
df_dir['BS_negative'] = bs_n
df_dir['RefPref'] = (df_dir['BS_positive'] > df_dir['BS_negative']).astype(int)

summary = df_dir.groupby('condition').agg(
    ref_pref_count=('RefPref', 'sum'),
    ref_pref_pct=('RefPref', lambda x: x.mean()*100),
    bertscore_f1=('BS_positive', 'mean')
).reset_index()

# Calculate Randomization Test P-value
steer_score = float(summary[summary['condition'] == '+v_steer']['ref_pref_pct'].iloc[0])
rand_scores = summary[summary['condition'].str.startswith('rand_seed_')]['ref_pref_pct'].values
p_val = float(1.0 + np.sum(rand_scores >= steer_score)) / float(len(rand_scores) + 1.0)

os.makedirs('outputs', exist_ok=True)
os.makedirs('results', exist_ok=True)
df_dir.to_csv('outputs/random_direction_outputs.csv', index=False)
summary.to_csv('results/random_direction_summary.csv', index=False)

print('========================================================================')
print(f'✅ +v_steer Reference Preference: {steer_score:.2f}%')
print(f'✅ 20 Random Vectors Distribution: Mean={np.mean(rand_scores):.2f}% ± {np.std(rand_scores):.2f}% (Min={np.min(rand_scores):.2f}%, Max={np.max(rand_scores):.2f}%)')
print(f'✅ Empirical Randomization Test P-Value: p = {p_val:.4f}')
print('========================================================================')
assert len(df_dir) == 22 * 500, 'Must have 22 conditions x 500 questions = 11,000 records'
print('✅ Automated Directional Control Assertions Passed 100%!')
